# 02 — EDA: The County Divide

**Questions explored:**
1. How does equitable share per capita vary across Kenya's 47 counties?
2. Which counties spend their money (high absorption) vs which don't?
3. Do counties that receive more per capita actually deliver better outcomes?
4. Who are the outliers — over-performers and under-performers?
5. Are the gaps between counties narrowing or widening over time?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA = os.path.join("..", "data", "clean")
county = pd.read_csv(os.path.join(DATA, "county_enriched.csv"))
composite = pd.read_csv(os.path.join(DATA, "composite_scores.csv"))

print(f"county_enriched  : {county.shape}")
print(f"composite_scores : {composite.shape}")
county.head()

## 1. Equitable Share Distribution Across Counties

> *Which counties get the most and least? How skewed is the distribution?*

In [ ]:
# Latest fiscal year available
latest_fy = county["fiscal_year"].sort_values().iloc[-1]
latest = county[county["fiscal_year"] == latest_fy].copy()
latest = latest.sort_values("allocation_mn", ascending=True)

fig, ax = plt.subplots(figsize=(10, 14))
colors = ["#E24A33" if v == latest["allocation_mn"].max() or v == latest["allocation_mn"].min()
          else "#348ABD" for v in latest["allocation_mn"]]
ax.barh(latest["county_name"], latest["allocation_mn"], color=colors)
ax.set_xlabel("Allocation (KSh Millions)")
ax.set_title(f"Equitable Share Allocation by County — {latest_fy}")
ax.axvline(latest["allocation_mn"].median(), color="gray", linestyle="--", label="Median")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean:   KSh {latest['allocation_mn'].mean():,.0f}M")
print(f"Median: KSh {latest['allocation_mn'].median():,.0f}M")
print(f"Std:    KSh {latest['allocation_mn'].std():,.0f}M")
print(f"Max/Min ratio: {latest['allocation_mn'].max() / latest['allocation_mn'].min():.1f}x")

## 2. Absorption Rates — Who Spends Their Money?

> *High allocation means nothing if the county can't spend it. Which counties have spending capacity?*

In [ ]:
# Average absorption rate per county across all FYs
avg_absorption = county.groupby("county_name")["absorption_rate_derived"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 14))
colors = ["#E24A33" if v < 70 else "#8EBA42" if v > 80 else "#348ABD" for v in avg_absorption]
ax.barh(avg_absorption.index, avg_absorption.values, color=colors)
ax.axvline(avg_absorption.median(), color="gray", linestyle="--", label=f"Median: {avg_absorption.median():.1f}%")
ax.set_xlabel("Average Absorption Rate (%)")
ax.set_title("County Budget Absorption Rate (Average 2013/14 – 2023/24)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nLowest absorbers:")
print(avg_absorption.head(5).to_string())
print(f"\nHighest absorbers:")
print(avg_absorption.tail(5).to_string())

## 3. Allocation Growth Over Time — Are Gaps Widening?

> *Do all counties grow at the same rate, or are some falling behind?*

In [ ]:
# Allocation trend for selected counties (top 5, bottom 5, and median)
alloc_pivot = county.pivot_table(index="fiscal_year", columns="county_name",
                                  values="allocation_mn")
ranked = alloc_pivot.iloc[-1].sort_values()
spotlight = list(ranked.head(3).index) + list(ranked.tail(3).index)
# Add a "middle" county
mid_idx = len(ranked) // 2
spotlight.append(ranked.index[mid_idx])

fig, ax = plt.subplots(figsize=(13, 6))
for county_name in spotlight:
    if county_name in alloc_pivot.columns:
        style = "-" if county_name in ranked.tail(3).index else "--"
        ax.plot(alloc_pivot.index, alloc_pivot[county_name], style,
                label=county_name, linewidth=2, marker="o", markersize=4)

ax.set_xlabel("Fiscal Year")
ax.set_ylabel("Allocation (KSh Millions)")
ax.set_title("Equitable Share Allocation Trend — Selected Counties")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Gini-like measure: coefficient of variation over time
cv = county.groupby("fiscal_year")["allocation_mn"].agg(["std", "mean"])
cv["cv"] = cv["std"] / cv["mean"] * 100
print("Coefficient of Variation (allocation) over time:")
print(cv["cv"].to_string())

## 4. Funds vs Outcomes — Does Money Buy Better Services?

> *Scatter plot: allocation vs indicators for counties where we have KDHS data.*

In [ ]:
# Filter to rows with indicator data
ind_cols = [c for c in county.columns if c.startswith("ind_")]
has_data = county.dropna(subset=ind_cols, how="all")

if len(has_data) > 0 and "ind_poverty_headcount" in has_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Allocation vs Poverty
    ax = axes[0]
    scatter_data = has_data.dropna(subset=["allocation_mn", "ind_poverty_headcount"])
    ax.scatter(scatter_data["allocation_mn"], scatter_data["ind_poverty_headcount"],
               c="steelblue", alpha=0.7, edgecolors="white", s=80)
    for _, row in scatter_data.iterrows():
        ax.annotate(row["county_name"], (row["allocation_mn"], row["ind_poverty_headcount"]),
                     fontsize=7, alpha=0.8)
    ax.set_xlabel("Allocation (KSh M)")
    ax.set_ylabel("Poverty Headcount (%)")
    ax.set_title("Allocation vs Poverty Rate")
    
    # Allocation vs Skilled Birth Attendance
    ax = axes[1]
    scatter_data2 = has_data.dropna(subset=["allocation_mn", "ind_skilled_birth_attendance"])
    ax.scatter(scatter_data2["allocation_mn"], scatter_data2["ind_skilled_birth_attendance"],
               c="darkorange", alpha=0.7, edgecolors="white", s=80)
    for _, row in scatter_data2.iterrows():
        ax.annotate(row["county_name"], (row["allocation_mn"], row["ind_skilled_birth_attendance"]),
                     fontsize=7, alpha=0.8)
    ax.set_xlabel("Allocation (KSh M)")
    ax.set_ylabel("Skilled Birth Attendance (%)")
    ax.set_title("Allocation vs Skilled Birth Attendance")
    
    plt.suptitle("Funds vs Outcomes (counties with KDHS data)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Insufficient indicator data for scatter plots.")

## 5. Composite Service Delivery Ranking

> *Which counties deliver the best and worst outcomes per shilling?*

In [ ]:
if len(composite) > 0 and "service_delivery_score" in composite.columns:
    # Filter to non-null scores
    scored = composite.dropna(subset=["service_delivery_score"])
    
    if len(scored) > 0:
        # Average score per county (across available years)
        avg_score = scored.groupby("county_name")["service_delivery_score"].mean().sort_values()
        
        fig, ax = plt.subplots(figsize=(10, 8))
        colors = ["#E24A33" if v < 30 else "#8EBA42" if v > 70 else "#348ABD" for v in avg_score]
        ax.barh(avg_score.index, avg_score.values, color=colors)
        ax.axvline(50, color="gray", linestyle="--", alpha=0.5, label="Midpoint (50)")
        ax.set_xlabel("Composite Service Delivery Score (0–100)")
        ax.set_title("County Service Delivery Ranking")
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print("\nTop 5:")
        print(avg_score.tail(5).to_string())
        print("\nBottom 5:")
        print(avg_score.head(5).to_string())
    else:
        print("No scored counties available.")
else:
    print("Composite scores not computed yet.")

## 6. Outlier Detection — Over- and Under-Performers

> *Counties that punch above or below their weight: high outcomes despite low funding (or vice versa).*

In [ ]:
# Merge composite score with allocation data
if len(composite) > 0:
    merged = composite.merge(
        county[["fiscal_year", "county_code", "county_name", "allocation_mn"]],
        on=["fiscal_year", "county_code", "county_name"],
        how="left"
    )
    scored = merged.dropna(subset=["service_delivery_score", "allocation_mn"])
    
    if len(scored) > 5:
        # Quadrant plot: allocation (x) vs score (y)
        med_alloc = scored["allocation_mn"].median()
        med_score = scored["service_delivery_score"].median()
        
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.scatter(scored["allocation_mn"], scored["service_delivery_score"],
                   c="steelblue", alpha=0.7, s=80, edgecolors="white")
        for _, row in scored.iterrows():
            ax.annotate(row["county_name"], 
                        (row["allocation_mn"], row["service_delivery_score"]),
                        fontsize=7, alpha=0.8)
        
        ax.axvline(med_alloc, color="gray", linestyle="--", alpha=0.5)
        ax.axhline(med_score, color="gray", linestyle="--", alpha=0.5)
        
        # Label quadrants
        ax.text(0.02, 0.98, "LOW FUNDS\nHIGH OUTCOMES\n(over-performers)", 
                transform=ax.transAxes, fontsize=9, color="green", va="top")
        ax.text(0.98, 0.02, "HIGH FUNDS\nLOW OUTCOMES\n(under-performers)", 
                transform=ax.transAxes, fontsize=9, color="red", ha="right")
        
        ax.set_xlabel("Allocation (KSh Millions)")
        ax.set_ylabel("Service Delivery Score (0–100)")
        ax.set_title("Funds vs Outcomes: Identifying Over/Under-Performers")
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough data for quadrant analysis.")
else:
    print("Composite scores required.")

## 7. Surprises, Gaps & Limitations

**Observations** (to refine after running cells):

- **CRA formula effect:** All counties move in lockstep because the equitable share percentages are fixed — the variation comes from *total* equitable share growth, not individual county changes.
- **Data sparsity:** Only 13 of 47 counties have KDHS indicator snapshots (2014 & 2022). This limits the funds-vs-outcomes analysis.
- **Absorption uniformity:** Because we applied aggregate absorption rates (from CoB annual reports) to all counties uniformly, the per-county absorption rates are identical within each FY. Real CoB data per county is needed.
- **WB indicators are empty:** The World Bank API was not reachable during data generation. Need to populate or use alternative sources.
- **No population data:** Per-capita calculations could not be computed without WB population figures.

**Next steps for Sprint 4:**
1. Source county-level CoB audit data for real absorption rates
2. Populate WB indicator values (or use manual KNBS Economic Survey data)
3. Expand KDHS coverage to all 47 counties (interpolate or use KIHBS 2015/16)
4. Consider using county population from 2019 Census for per-capita metrics